# Experimenting
Testing the clean dataset with on the model and algorithm scripts. (unofficially)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pygam import LogisticGAM, s, f
from sklearn.preprocessing import StandardScaler

from algorithms.gradient_descent import gradient_descent
from models.logistic import logistic_loss, logistic_grad, sigmoid

In [15]:
from models.logistic import logistic_loss, logistic_grad, sigmoid
from algorithms.gradient_descent import gradient_descent

In [2]:
# load the clean data
df = pd.read_csv("../data/framingham_cleaned.csv")
df.head()

,male,age,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,TenYearCHD
0,1,39,0,0.0,0.0,0,0,0,195.0,106.0,70.0,26.97,80.0,0
1,0,46,0,0.0,0.0,0,0,0,250.0,121.0,81.0,28.73,95.0,0
2,1,48,1,20.0,0.0,0,0,0,245.0,127.5,80.0,25.34,75.0,0
3,0,61,1,30.0,0.0,0,1,0,225.0,150.0,95.0,28.58,65.0,1
4,0,46,1,23.0,0.0,0,0,0,285.0,130.0,84.0,23.10,85.0,0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4090 entries, 0 to 4089
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   male             4090 non-null   int64  
 1   age              4090 non-null   int64  
 2   currentSmoker    4090 non-null   int64  
 3   cigsPerDay       4090 non-null   float64
 4   BPMeds           4090 non-null   float64
 5   prevalentStroke  4090 non-null   int64  
 6   prevalentHyp     4090 non-null   int64  
 7   diabetes         4090 non-null   int64  
 8   totChol          4090 non-null   float64
 9   sysBP            4090 non-null   float64
 10  diaBP            4090 non-null   float64
 11  BMI              4090 non-null   float64
 12  heartRate        4090 non-null   float64
 13  TenYearCHD       4090 non-null   int64  
dtypes: float64(7), int64(7)
memory usage: 447.5 KB


In [4]:
df.isna().sum()

male               0
age                0
currentSmoker      0
cigsPerDay         0
BPMeds             0
prevalentStroke    0
prevalentHyp       0
diabetes           0
totChol            0
sysBP              0
diaBP              0
BMI                0
heartRate          0
TenYearCHD         0
dtype: int64

In [5]:
feature_cols = ["male", "age", "currentSmoker", "cigsPerDay", "BPMeds",
    "prevalentStroke", "prevalentHyp", "diabetes",
    "totChol", "sysBP", "diaBP", "BMI", "heartRate"]

target_col = "TenYearCHD"

In [12]:
x_raw = df[feature_cols].to_numpy()
y = df[target_col].to_numpy()

print("Raw X shape:", x_raw.shape)
print("Raw y shape:", y.shape)

Raw X shape: (4090, 13)
Raw y shape: (4090,)


In [9]:
# scale features so data plays nice with all the algorithms
scaler = StandardScaler()
X_scaled = scaler.fit_transform(x_raw)

In [13]:
# adding a bias column to the dataset specifically for Newton/BFGS
ones = np.ones((X_scaled.shape[0], 1))
X = np.hstack([X_scaled, ones])
print("X with bias shape:", X.shape)

X with bias shape: (4090, 14)


In [11]:
# weight vector with the same length as the columns in X -> needed for optimizing
w0 = np.zeros(X.shape[1])

In [16]:
# run gradient descent on full dataset (just a test)
print("\n=== Gradient Descent sanity test (logistic on cleaned data) ===")
print("Initial loss:", logistic_loss(w0, X, y))

w_gd, history_gd = gradient_descent(w0=w0, loss_fn=logistic_loss, grad_fn=logistic_grad, X=X, y=y, lr=0.01, iters=500)

print("Final loss:", history_gd[-1])
print("Iterations:", len(history_gd))
print("First 5 losses:", history_gd[:5])
print("Last 5 losses:", history_gd[-5:])


=== Gradient Descent sanity test (logistic on cleaned data) ===
Initial loss: 0.6931471785599453
Final loss: 0.43053360205291397
Iterations: 500
First 5 losses: [np.float64(0.6916740241457151), np.float64(0.6902101744227624), np.float64(0.6887555531967168), np.float64(0.6873100849638024), np.float64(0.6858736949066436)]
Last 5 losses: [np.float64(0.4311496552600017), np.float64(0.4309948303193974), np.float64(0.43084054801921956), np.float64(0.4306868060347279), np.float64(0.43053360205291397)]


In [17]:
def predict_proba(X, w):
    return sigmoid(X @ w)

def predict_class(X, w, threshold=0.5):
    return (predict_proba(X, w) >= threshold).astype(int)


In [18]:
y_pred = predict_class(X, w_gd)
acc = (y_pred == y).mean()
print("Training accuracy (full data, no split)", acc)

Training accuracy (full data, no split) 0.8533007334963325
